# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">8주차 · FastAPI 로 서빙 — 투자자 번호를 보내면 추천을 응답하는 API</mark>

## 한눈에 보기

| 질문 | 답 |
|---|---|
| **오늘 무엇을 하나요?** | 7주차에 만든 추천 파일(`recommendations.csv`)을 읽어, **투자자 번호를 받으면 그 투자자의 추천을 응답하는 API** 를 켜고 요청을 보내 봅니다 |
| **그 결과로 무엇이 생기나요?** | 주소 하나(`/recommend/U0001`)로 추천 10개를 받을 수 있는 **서버**가 동작합니다 |
| **왜 하나요?** | 앱 · 상담 화면 · 챗봇처럼 추천을 쓰는 프로그램이 여러 개일 때, 각자 파일을 읽지 않고 **같은 주소로 요청해서, API 가 읽은 같은 결과 파일의 추천**을 받게 하려는 것입니다 |
| **추천 내용이 바뀌나요?** | 아니요. API 는 학습하지 않고, 7주차 파일의 추천을 **그대로 응답**합니다. 바뀌는 것은 추천을 **전달하는 방법**입니다 |

---

## 왜 필요한가 — 예시 상황

증권사의 모바일 앱 · 상담 직원 화면 · 챗봇이 모두 투자자에게 추천을 보여 주려 한다고 해 봅시다. 세 프로그램이 7주차 CSV 파일을 **각자 직접 읽으면** 다음 문제가 생깁니다.

| 문제 | 실제로 생기는 일 |
|---|---|
| 같은 추천을 받지 못한다 | 프로그램마다 복사해 둔 파일의 날짜가 다를 수 있어서, 같은 고객에게 서로 다른 추천이 나갑니다 |
| 다른 고객의 정보까지 넘어간다 | 파일에는 **300명 전원**의 추천과 투자 성향이 들어 있습니다. 한 사람의 추천이 필요한 앱에 전체를 줘야 합니다 |
| 파일 형식에 묶인다 | 파일 위치 · 열 순서가 바뀌면 모든 프로그램을 고쳐야 합니다 |
| 잘못된 요청을 구분하지 못한다 | 없는 투자자 번호나 잘못된 값이 와도 알려 줄 방법이 없습니다 |

**오늘 해결하는 방법**
- 파일은 **API 한 곳**에서만 읽습니다. 모든 프로그램은 파일이 아니라 API 에 요청합니다
- API 는 요청한 **한 사람의 추천만** 응답합니다
- 잘못된 요청에는 **오류 번호와 이유**를 응답합니다

---

## 오늘 만드는 것 — 미리 보기

요청 한 줄을 보내면 응답이 JSON 으로 돌아옵니다 (`k=2` 는 2개만 받겠다는 뜻입니다).

```
요청   GET http://localhost:8765/recommend/U0001?k=2

응답   {"user_id": "U0001", "persona": "안정형",
        "recommendations": [
          {"rank": 1, "item_id": "I002", "item_name": "TIGER 미국배당다우존스", "risk_level": 2, "score": 0.6211},
          {"rank": 2, "item_id": "I043", "item_name": "Vanguard FTSE Developed Markets ETF", "risk_level": 3, "score": 0.4705}]}
```

- API 코드는 **`recommend_api.py` 파일 하나**에 들어 있습니다. 이 노트북은 그 API 를 **켜고 → 요청을 보내고 → 응답을 확인**합니다.
- 7주차 결과 파일을 읽을 뿐이라서, 추천 방법 · 점수 기준은 그대로입니다.

---

### 오늘의 구성 — 50분 (PPT + 노트북)

| 시간 | 어디서 | 하는 일 |
|---|---|---|
| 개념 30분 | PPT | 클라이언트 · 서버 · 요청과 응답 · 엔드포인트 · HTTP 메서드 · 매개변수 · JSON · 상태 코드 · 입력 검증 · 일괄 추천과 실시간 추천 · 인증과 권한 |
| 실습 20분 | **이 노트북** | 모두 실행 → 8.1 API 코드 읽기 → 8.2 서버 켜기 → 8.3 요청 보내기 → 8.4 Swagger (강사 시연) → 8.5 오류 응답 → 🗣 함께 보기 |

### 이 노트북 읽는 법 — 🎯 와 📖

- **🎯 수업** — 수업 시간에 셀을 실행하고 **결과를 함께 읽는** 절입니다
  - 8.1 \~ 8.5 가 여기에 속합니다
- **📖 혼자 읽기** — 수업에서는 건너뛰고 **복습할 때 읽는** 절입니다
  - 8.6 \~ 8.10 이 여기에 속합니다

⏱ **실습을 시작하면 먼저 메뉴의 `런타임 → 모두 실행` 을 눌러 두세요.** 7주차 결과 파일 만들기와 서버 켜기까지 20초쯤 걸립니다. 그동안 위에서부터 결과를 읽으면 됩니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/08_api_serving.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

> ⚠ **오류가 나거나 지난번 실행 내용이 남아 있는 것 같으면** `런타임 → 세션 다시 시작` 을 누른 뒤 `런타임 → 모두 실행` 을 누르세요. 서버도 함께 새로 켜집니다.

---

**이 노트북이 쓰는 것**

| 무엇 | 어디에 | 쓰임 |
|---|---|---|
| `recommend_batch.py` | `notebooks/` 폴더 | 7주차 스크립트 — 결과 파일 `output/recommendations.csv` 를 만든다 |
| `recommend_api.py` | `notebooks/` 폴더 | **오늘의 산출물** — 결과 파일을 읽어 추천을 응답하는 API |
| `items.csv` · `users.csv` · `interactions.csv` | `data/` 폴더 | 7주차 스크립트가 읽는 데이터 |

- 모두 **파일**입니다. 아래 셀이 저장소를 `/content/recsys` 에 내려받으면 Colab 왼쪽의 📁 에서 볼 수 있습니다. 처음에는 `sample_data` 만 보이고, 셀을 실행한 뒤 ↻ 를 누르면 `recsys` 폴더가 생깁니다.

⚠ **지난주 노트북을 먼저 실행할 필요가 없습니다.** 7주차 결과 파일도 이 노트북이 다시 만듭니다.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받고, API 를 만드는 도구(FastAPI · uvicorn)가 있는지 확인한다
# 왜 하나 — Colab 은 열 때마다 새 환경이라 파일이 없다. 내 컴퓨터에서 실행할 때는 자료를 내려받는 단계가 필요 없다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    if not os.path.exists("/content/recsys/notebooks"):                     # 내려받기에 실패했으면 이유를 알리고 멈춘다
        raise RuntimeError("실습 자료를 받지 못했습니다. 인터넷 연결을 확인하고 이 셀을 다시 실행하세요.")
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

try:
    import fastapi, uvicorn                          # 이미 설치되어 있으면 아무것도 하지 않는다
except ImportError:                                  # 없으면 설치한다 (10초쯤 걸린다)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"], check=True)   # 실패하면 여기서 멈춘다
    print("FastAPI · uvicorn 을 설치했습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 나오는 말</mark>

**여기서 모두 외울 필요는 없습니다.** 괄호 안의 절에서 예와 함께 다시 설명합니다. 지금은 이름만 읽고, 8.1 부터 하나씩 직접 확인합니다. **「언제 유용한가」 열**은 그 말을 실제로 쓰는 장면입니다.

**① 요청과 응답**

| 말 | 뜻 | 언제 유용한가 | 어디서 |
|---|---|---|---|
| **API** (Application Programming Interface) | 다른 프로그램의 요청을 받아 **정해진 형식으로 응답**하는 기능 | 앱 · 상담 화면 · 챗봇이 **같은 API 에 요청**해서 추천을 받을 때. 내부를 바꿔도 요청과 응답의 형식이 그대로면 쓰는 쪽은 고칠 필요가 없습니다 | 8.1 |
| **클라이언트 (client) · 서버 (server)** | 요청을 **보내는** 쪽 · 요청을 **받아 응답하는** 쪽 | 오류가 났을 때 **요청을 보낸 쪽과 처리한 쪽 중 어디가 원인인지** 나눠서 살펴볼 때 | 8.2 |
| **요청 (request) · 응답 (response)** | 클라이언트가 서버에 보내는 것 · 서버가 클라이언트에 보내는 것 | 무엇을 보냈고 무엇을 받았는지 살펴서 **문제의 원인을 좁힐** 때 | 8.3 |
| **엔드포인트 (endpoint)** | 요청을 받는 **주소 하나**. 오늘은 `/health` 와 `/recommend/{user_id}` | 기능마다 주소를 나눠 두고 다른 팀에 「추천은 이 주소를 부르세요」라고 **알려 줄 때** | 8.1 |
| **HTTP 메서드 (method)** | 요청의 **종류**. `GET` = 가져오기(조회), `POST` = 데이터를 보내 처리(예 — 저장)를 요청하기 | 추천을 받을 때는 `GET`, 고객이 종목을 담은 기록을 보낼 때는 `POST` 로 **일을 구분**할 때. 지원하지 않는 메서드로 요청하면 405 로 응답합니다 | 8.5 |
| **매개변수 (parameter)** | 주소 안에 값을 넣는 방법. **경로 매개변수**(`/recommend/U0001` 의 `U0001`)와 **쿼리 매개변수**(`?k=3` 의 `k`) | 투자자 번호는 **경로**에, 받을 개수는 **쿼리**에 넣어서 사람마다 · 개수마다 다른 응답을 받을 때 | 8.3 · 8.6 |
| **JSON** (JavaScript Object Notation) | 응답에 쓰는 글자 형식. 파이썬 딕셔너리와 거의 같은 모양 | 파이썬 · 웹 · 모바일 앱처럼 **서로 다른 프로그램이 같은 형식**으로 데이터를 주고받을 때 | 8.3 |

**② 응답의 약속**

| 말 | 뜻 | 언제 유용한가 | 어디서 |
|---|---|---|---|
| **상태 코드 (status code)** | 응답 맨 앞의 결과 번호. **2xx** 성공 · **4xx** 요청한 쪽의 문제 · **5xx** 서버 쪽의 문제 | 앱이 번호만 보고 **화면을 정할 때**(예 — 200 이면 추천 표시, 404 면 「고객을 찾을 수 없습니다」 안내). 접속 기록에서 오류가 늘었는지 볼 때도 씁니다 | 8.5 |
| **입력 검증 (validation)** | 요청 값이 규칙에 맞는지 서버가 **먼저 검사**하는 것 | 허용 범위를 벗어난 값(예 — 최대 10개로 정했는데 `k=1000`)이 들어와 **이상한 응답이 나가거나 서버에 부담**이 되는 것을 막을 때 | 8.5 |
| **응답 규격 (response schema)** | 응답의 **항목 이름과 형식**을 정해 둔 약속. FastAPI 는 코드에 적은 이 규격으로 응답을 검사하고 설명서에도 기록합니다. 7주차의 결과 파일 규격(output schema)과 같은 생각 | 앱 개발팀이 **응답의 모양만 보고 화면을 만들** 때. 모양이 바뀌면 앱이 깨지므로 미리 약속해 둡니다 | 8.6 |

**③ 도구와 서빙**

| 말 | 뜻 | 언제 유용한가 | 어디서 |
|---|---|---|---|
| **FastAPI** · **Swagger** | 파이썬으로 API 를 만드는 도구 · API 설명서(OpenAPI)를 화면으로 보여 주고 시험도 해 볼 수 있는 도구(Swagger UI, 주소 `/docs`) | **짧은 코드**로 API 를 만들고, 설명서를 따로 쓰지 않고 `/docs` 주소만 알려 줄 때. 코드 없이 시험해 볼 때 | 8.1 · 8.4 |
| **일괄 추천 · 실시간 추천** | 미리 계산해 둔 추천을 응답하는 방식 · 요청이 올 때 계산하는 방식 | 반복되는 요청에 빠르게 응답해야 하면 일괄 추천, **방금 한 행동을 바로 반영**해야 하면 실시간 추천을 고려합니다 | 8.7 |
| **지연 시간 (latency)** | 요청을 보낸 뒤 응답이 올 때까지 걸리는 시간 | 「화면이 몇 초 안에 떠야 한다」 같은 요구를 **숫자로 확인**하고, 방식을 비교할 때 | 8.7 |
| **인증 (authentication) · 권한 (authorization)** | 요청한 사람이 **누구인지** 확인 · 그 사람이 **접근할 수 있는 정보 · 기능의 범위**를 확인 | 투자 성향 같은 **고객 개인정보**를 본인과 권한이 있는 직원만 보게 할 때. 금융 API 에서는 특히 중요합니다 | 8.8 |

---

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 준비 — 7주차 결과 파일 만들기</mark>

**여기서 하는 일** — 7주차 스크립트를 한 번 실행해 결과 파일을 만듭니다. API 가 읽을 파일입니다.

- Colab 은 열 때마다 새 환경이라서, 지난주에 만든 파일이 남아 있지 않습니다.

In [2]:
# 이 셀에서 하는 일 — 7주차 스크립트로 결과 파일을 만든다
# 왜 하나 — 오늘 API 가 읽을 파일이 output/recommendations.csv 이다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구
sys.path.insert(0, ".")                     # 지금 폴더에서 recommend_batch.py · recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 단계 안쪽 폴더(notebooks)도 찾게 한다

!python recommend_batch.py

① 데이터 점검 — 여섯 가지 모두 0건 · 통과


② 학습 — 상호작용 기록 8,086건 · 투자자 300명


③ 일괄 추천 — 3,000줄
④ 규격 검사 — 문제 0개 · 통과
⑤ 저장 — output/recommendations.csv · 300명 × 10개


**결과 읽는 법**

**결론: 마지막 줄 「⑤ 저장」이 나오면 API 가 읽을 결과 파일이 준비된 것입니다.**
- 7주차 7.3 과 같은 다섯 줄입니다 — 데이터 점검 통과 → 상호작용 8,086건으로 학습 → 3,000줄 → 규격 검사 통과 → 저장

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 8.1 API 코드 읽기 — 요청 하나를 처리하는 다섯 단계</mark>

**흐름에서의 위치:** 결과 파일 → **API 코드** → (서버로 켜기, 8.2) → 요청 · 응답 (8.3). 아직 켜지 않고 코드만 읽습니다.

**8.1 에서 하는 일** — `recommend_api.py` 의 코드를 화면에 띄워서, **엔드포인트 두 개**가 무엇을 받고 무엇을 응답하는지 읽습니다.

**왜 하나**
- API 가 요청을 받아 어떻게 응답하는지가 이 파일 안에 있습니다. 코드는 짧아서 「어떤 주소에서 무엇을 응답하는지」는 비개발자도 읽을 수 있습니다.

**코드의 구성**

| 부분 | 하는 일 | 언제 실행되나 |
|---|---|---|
| 결과 파일 읽기 | 7주차 파일을 읽어 메모리에 올려 둔다. 파일이 없거나 규격과 다르면 **켜지지 않는다** | API 가 **켜질 때 한 번** |
| 응답 규격 `RecommendItem` · `RecommendResponse` | 응답의 항목 이름과 형식을 정한다 | 〃 |
| 엔드포인트 `/health` | 켜져 있는지, 몇 줄을 읽었는지 응답한다 | 요청이 올 때마다 |
| 엔드포인트 `/recommend/{user_id}` | 그 투자자의 추천을 응답한다 | 요청이 올 때마다 |

**`/recommend/{user_id}` 가 요청 하나를 처리하는 다섯 단계**
1. **고른다** — 결과 파일에서 그 투자자의 줄만 고른다
2. **없으면 404** — 줄이 하나도 없으면 「추천이 없는 투자자」라는 오류로 응답하고 끝낸다
3. **자른다** — 순위 순서로 정렬해서 위에서 `k` 개만 남긴다
4. **모양을 맞춘다** — 한 줄씩 응답 규격(`RecommendItem`)에 맞춘다
5. **응답한다** — 투자자 번호 · 투자 성향 · 추천 목록을 응답한다. FastAPI 가 JSON 으로 바꿔서 보낸다

**코드를 읽을 때 알아 둘 표시**
- `@app.get("/health")` — 「`/health` 주소로 **GET 요청**이 오면 바로 아래 함수를 실행하라」는 표시입니다. `@` 로 시작하는 줄을 **데코레이터(decorator)** 라고 합니다
- `/recommend/{user_id}` — 중괄호 자리에 투자자 번호가 들어갑니다. `/recommend/U0001` 이면 `user_id` 가 `"U0001"` 입니다
- `k: int = Query(10, ge=1, le=10)` — 주소 뒤 `?k=5` 로 받을 개수를 정합니다. 안 적으면 10, 1 이상 10 이하만 허용합니다 (`ge` 이상 · `le` 이하)
- `raise HTTPException(status_code=404, ...)` — 오류 번호와 이유로 응답하고 함수를 끝냅니다

In [3]:
# 이 셀에서 하는 일 — API 파일의 코드를 화면에 띄운다
# 왜 하나 — 엔드포인트 두 개와 요청 하나를 처리하는 다섯 단계를 직접 읽는다

with open("recommend_api.py", encoding="utf-8") as 파일:   # with = 다 읽으면 파일을 저절로 닫는다
    코드 = 파일.read()
print(코드[코드.index("결과_파일 ="):])                      # 맨 위 설명은 건너뛰고 코드 부분만 보여 준다

결과_파일 = Path(__file__).resolve().parent / "output" / "recommendations.csv"   # 7주차가 저장한 파일
필요한_열 = ["user_id", "rank", "item_id", "item_name", "risk_level", "persona", "score"]   # 7주차 결과 파일 규격(output schema)


# ── 켜질 때 한 번만 실행된다 ──────────────────────────────────────────────

if not 결과_파일.exists():                                      # 파일이 없으면 API 를 켜지 않고 이유를 알려 준다
    raise FileNotFoundError(f"결과 파일이 없습니다: {결과_파일}\n7주차의 python recommend_batch.py 를 먼저 실행하세요")

추천표 = pd.read_csv(결과_파일, encoding="utf-8-sig")           # 3,000줄을 메모리에 올려 둔다. 요청마다 파일을 다시 읽지 않는다

if 추천표.empty:                                               # 줄이 하나도 없는 파일이면 멈춘다
    raise ValueError("결과 파일이 비어 있습니다")

빠진_열 = [열 for 열 in 필요한_열 if 열 not in 추천표.columns]
if 빠진_열:                                                    # 규격과 다른 파일이면 잘못된 응답이 나가기 전에 멈춘다
    raise ValueError(f"결과 파일에 필요한 열이 없습니다: {빠진_열}")


# ── 응답의 모양 (response schema) — Swagger 화면에도 그대로 나온다 ──────────

class RecommendItem(BaseModel):          # 영어 이름: 

**결과 읽는 법**

**결론: `recommend` 함수 안에 학습하는 줄이 없습니다. 파일에서 고르고 · 자르고 · 응답할 뿐입니다.**
- 「켜질 때 한 번 실행」 부분(`추천표 = pd.read_csv(...)`)과 「요청마다 실행」 부분(`def health` · `def recommend`)이 나뉘어 있습니다
- `# 1)` \~ `# 5)` 주석이 위의 다섯 단계와 같은 순서입니다
- 파일이 없으면 API 가 켜지지 않습니다. 그 이유(`FileNotFoundError`)는 8.2 의 서버 셀이 화면에 보여 줍니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 8.2 서버 켜기 — API 를 켜 두고 켜져 있는지 확인</mark>

**흐름에서의 위치:** API 코드 → **서버로 켜기** → 요청 · 응답 (8.3).

**8.2 에서 하는 일** — API 를 **서버**로 켜 두고, `/health` 에 요청을 보내 「켜져 있다」는 응답을 받습니다.

**왜 하나**
- API 는 코드 파일만 있다고 동작하지 않습니다. 누군가 **켜 두어야** 요청을 받을 수 있습니다.
- 실제 서비스에서는 「API 가 꺼졌는지」를 몇 초마다 확인합니다. `/health` 가 그 확인용 주소입니다.

**클라이언트와 서버**
- **서버 (server)** — 계속 켜져 있으면서 요청을 기다리고, 요청이 오면 응답하는 프로그램. 오늘은 API 가 서버입니다
- **클라이언트 (client)** — 서버에 요청을 보내는 프로그램. 오늘은 **이 노트북**이 클라이언트입니다
- 앱 · 상담 화면 · 챗봇도 모두 클라이언트입니다

**주소 읽는 법** — `http://localhost:8765/health`

| 부분 | 뜻 |
|---|---|
| `http://` | 요청과 응답을 주고받는 약속(HTTP) |
| `localhost` | **지금 코드가 실행되는 컴퓨터**. Colab 에서는 Colab 런타임입니다. 오늘 API 는 이 안에서만 부를 수 있습니다 |
| `8765` | **포트 (port)**. 한 컴퓨터 안에서 프로그램을 구분하는 번호 |
| `/health` | 엔드포인트. 이 서버 안에서 어느 기능을 부를지 |

**순서**
1. 먼저 `/health` 에 요청을 보내 본다 — 이미 켜져 있으면 다시 켜지 않는다
2. 꺼져 있으면 **uvicorn**(FastAPI 코드를 서버로 켜 주는 도구)으로 `recommend_api.py` 를 **백그라운드**로 켠다. 셀이 끝나도 계속 실행된다는 뜻이다
3. 켜질 때까지 0.5초마다 다시 요청한다 (최대 20초)
4. 켜지면 `/health` 의 응답을 보여 준다

In [4]:
# 이 셀에서 하는 일 — API 서버를 백그라운드로 켜고, /health 로 켜져 있는지 확인한다
# 왜 하나 — 서버가 켜져 있어야 요청을 보낼 수 있다

import subprocess                         # 다른 프로그램을 켜는 도구
import time                               # 기다리는 도구
import requests                           # 요청을 보내는 도구 (이 노트북이 클라이언트가 된다)
import pandas as pd                       # 표를 다루는 도구

포트 = 8765                                   # 오늘 API 서버의 번호
주소 = f"http://localhost:{포트}"             # 이 컴퓨터 안의 8765 번 서버
로그_파일 = "output/api_log.txt"              # 서버가 남기는 글자(접속 기록 · 오류)를 모아 두는 파일

def 켜져_있나():
    """/health 에 요청을 보내 200 이 오면 True."""
    try:
        return requests.get(주소 + "/health", timeout=1).status_code == 200
    except requests.exceptions.RequestException:   # 아직 안 켜졌으면 연결 오류가 난다
        return False

if 켜져_있나():                                          # 이미 켜져 있으면 다시 켜지 않는다
    print("이미 켜져 있는 서버를 사용합니다. API 코드를 바꿨다면 런타임 → 세션 다시 시작 후 모두 실행하세요.")
else:
    with open(로그_파일, "w", encoding="utf-8") as 로그:
        subprocess.Popen([sys.executable, "-m", "uvicorn", "recommend_api:app", "--port", str(포트)],
                         stdout=로그, stderr=로그)         # 서버가 찍는 글자를 로그 파일에 모은다
    for _ in range(40):                                  # 최대 20초 기다린다
        if 켜져_있나():
            break
        time.sleep(0.5)
    if not 켜져_있나():                                    # 20초 안에 켜지지 않았으면 이유를 보여 주고 멈춘다
        with open(로그_파일, encoding="utf-8") as 로그:
            print(로그.read()[-1500:])                      # 로그의 마지막 부분에 오류 이유가 있다
        raise RuntimeError("API 서버가 켜지지 않았습니다. 위 로그의 마지막 줄이 이유입니다.")

응답 = requests.get(주소 + "/health", timeout=5)
print("상태 코드 :", 응답.status_code)
print("응답 :", 응답.json())

상태 코드 : 200
응답 : {'status': 'ok', 'rows': 3000}


**결과 읽는 법**

**결론: 서버가 켜졌습니다. 상태 코드 200 과 `rows: 3000` 은 「켜져 있고, 결과 파일 3,000줄을 읽었다」는 뜻입니다.**
- **상태 코드 200** — 요청이 성공했습니다
- **응답 `{'status': 'ok', 'rows': 3000}`**
  - `status: ok` — 서버가 켜져 있다
  - `rows: 3000` — 켜질 때 결과 파일 3,000줄(300명 × 10개)을 읽었다
- 이제 서버는 셀이 끝나도 백그라운드에서 계속 요청을 기다립니다
  - Colab 런타임이 종료되면 서버도 꺼집니다(창을 닫거나 한동안 쓰지 않으면 종료될 수 있습니다). 그때는 `런타임 → 모두 실행` 을 다시 누릅니다

**자세히**
- `subprocess.Popen` 은 다른 프로그램을 **켜 두고 기다리지 않는** 도구입니다. 7주차의 `!python` 은 스크립트가 끝날 때까지 기다렸지만, 서버는 끝나지 않는 프로그램이라 기다리면 안 됩니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 8.3 요청 보내기 — U0001 의 추천을 받는다</mark>

**흐름에서의 위치:** 클라이언트(노트북) → **요청** → 서버(API) → **응답** → 클라이언트.

**8.3 에서 하는 일** — `/recommend/U0001` 에 **GET 요청**을 보내 JSON 응답을 받고, 표로 바꿔 봅니다. 7주차 파일과 같은지도 확인합니다.

**왜 하나**
- 오늘의 핵심 장면입니다. 다른 프로그램은 파일을 몰라도 **주소 하나**로 추천을 받습니다.
- 모든 프로그램이 같은 API 에 요청하면 API 가 읽은 **같은 결과 파일의 같은 추천**을 받습니다. 각자 파일을 읽으면 누군가는 어제 파일을, 누군가는 오늘 파일을 읽을 수 있습니다.
- 추천 방법이 나중에 바뀌어도 **주소와 응답의 모양만 그대로면** 앱은 고칠 필요가 없습니다.

**요청과 응답**

| | 내용 | 이 셀에서 |
|---|---|---|
| 요청 | 메서드 `GET` + 주소 `/recommend/U0001` | `requests.get(주소 + "/recommend/U0001")` |
| 응답 | 상태 코드 + JSON 글자 | `응답.status_code` · `응답.json()` |

**JSON 과 파이썬 딕셔너리**

| JSON (API 가 보낸 글자) | 파이썬에서 받으면 |
|---|---|
| `{"user_id": "U0001"}` | 딕셔너리 `{"user_id": "U0001"}` |
| `[{"rank": 1}, {"rank": 2}]` | 딕셔너리들의 목록 |

- `응답.json()` — 받은 JSON 글자를 파이썬 딕셔너리로 바꿉니다
- `pd.DataFrame(목록)` — 딕셔너리들의 목록을 표로 바꿉니다

In [5]:
# 이 셀에서 하는 일 — U0001 의 추천을 요청하고, 받은 JSON 을 표로 본다
# 왜 하나 — 다른 프로그램이 주소 하나로 추천을 받는 장면을 직접 본다

응답 = requests.get(주소 + "/recommend/U0001", timeout=5)   # U0001 의 추천을 달라는 GET 요청
답 = 응답.json()                                          # JSON 글자 → 파이썬 딕셔너리

print("보낸 요청 :", 응답.request.method, 응답.url)
print("상태 코드 :", 응답.status_code)
print("응답에 든 것 :", list(답.keys()))
print(f"투자자 {답['user_id']} · 투자 성향 {답['persona']} · 추천 {len(답['recommendations'])}개")
pd.DataFrame(답["recommendations"])                        # 추천 목록 → 표

보낸 요청 : GET http://localhost:8765/recommend/U0001
상태 코드 : 200
응답에 든 것 : ['user_id', 'persona', 'recommendations']
투자자 U0001 · 투자 성향 안정형 · 추천 10개


,rank,item_id,item_name,risk_level,score
0,1,I002,TIGER 미국배당다우존스,2,0.6211
1,2,I043,Vanguard FTSE Developed Markets ETF,3,0.4705
2,3,I014,KT,2,0.3974
3,4,I011,KODEX 단기채권,1,0.3722
4,5,I021,Verizon Communications,2,0.3690
5,6,I015,LG유플러스,2,0.3645
6,7,I033,TIGER 은행고배당플러스TOP10,2,0.3356
7,8,I020,Johnson & Johnson,2,0.3127
8,9,I035,TIGER 200,3,0.3036
9,10,I055,Apple,3,0.2801


**결과 읽는 법**

**결론: 주소 하나로 U0001 의 추천 10개를 받았습니다. 이 셀은 파일을 열지 않았습니다 — 파일은 API 가 켜질 때 읽어 두었습니다.**
- **보낸 요청** — 메서드 `GET`, 주소 `.../recommend/U0001`
- **상태 코드 200** — 성공입니다
- **응답에 든 것** — `user_id` · `persona` · `recommendations` 세 가지입니다. 8.1 의 응답 규격(`RecommendResponse`)과 같습니다
- 아래 표는 `recommendations` 를 표로 바꾼 것입니다. U0001(안정형)의 추천 **10개**이고, `risk_level` 은 모두 **3 이하**입니다
  - 1등은 TIGER 미국배당다우존스(위험도 2, 점수 0.6211)입니다. 7주차 7.4 에서 본 U0001 의 1등과 같습니다

In [6]:
# 이 셀에서 하는 일 — API 의 응답이 7주차 결과 파일과 같은지 비교한다
# 왜 하나 — API 가 추천을 바꾸지 않고 파일의 내용을 그대로 응답하는지 확인한다

파일 = pd.read_csv("output/recommendations.csv", encoding="utf-8-sig")
파일_U0001 = list(파일[파일["user_id"] == "U0001"].sort_values("rank")["item_id"])   # 파일에서 읽은 U0001 의 10개 (순위 순서)

API_U0001 = []
for 추천 in 답["recommendations"]:                                       # API 가 응답한 10개
    API_U0001.append(추천["item_id"])

print("파일 :", 파일_U0001)
print("API  :", API_U0001)
print("같은가 :", 파일_U0001 == API_U0001)

파일 : ['I002', 'I043', 'I014', 'I011', 'I021', 'I015', 'I033', 'I020', 'I035', 'I055']
API  : ['I002', 'I043', 'I014', 'I011', 'I021', 'I015', 'I033', 'I020', 'I035', 'I055']
같은가 : True


**결과 읽는 법**

**결론: API 의 응답이 7주차 파일과 순서까지 같습니다(`True`). API 는 추천을 새로 계산하지 않습니다.**
- 두 줄의 종목 번호 10개가 같습니다
- 만들어 둔 추천을 약속된 모양으로 전달하는 것을 **서빙(serving)** 이라고 합니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 8.4 Swagger — 코드 없이 버튼으로 요청하기 (강사 시연)</mark>

**흐름에서의 위치:** 8.3 은 코드로 요청했습니다. 8.4 는 같은 요청을 **화면의 버튼**으로 보냅니다.

**8.4 에서 하는 일** — FastAPI 가 자동으로 만든 **Swagger 화면**(`/docs`)에서 버튼으로 U0001 의 추천을 요청해 봅니다.

> 🎥 **수업에서는 강사가 화면으로 보여 줍니다.** 복습할 때 직접 따라 해 보세요.

**왜 하나**
- API 를 쓰는 사람(다른 팀 개발자)은 「어떤 주소에 무엇을 보내면 무엇이 오는지」를 알아야 합니다. Swagger 가 그 **설명서**입니다.
- 코드를 쓰지 않아도 버튼으로 **직접 시험**해 볼 수 있습니다.
- 설명서를 사람이 따로 쓰면 코드가 바뀔 때 설명서가 낡습니다. FastAPI 는 설명서(OpenAPI)를 **코드에서 자동으로** 만들고, Swagger UI 가 그것을 화면으로 보여 줍니다. 그래서 코드가 바뀌면 설명서도 함께 바뀝니다.
- 코드의 함수 설명(`"""..."""`)과 `description=` 이 Swagger 화면에 그대로 나옵니다. 설명서의 표준 형식을 **OpenAPI** 라고 합니다.

**어떻게 따라 하나**
1. 아래 셀을 실행하면 셀 아래에 Swagger 화면이 나타납니다 (Colab). 작으면 「새 탭으로 열기」 링크를 누릅니다
2. `GET /recommend/{user_id}` 줄을 누릅니다
3. 오른쪽 **`Try it out`** 버튼을 누릅니다
4. `user_id` 입력란에 `U0001` 을 적고 **`Execute`** 를 누릅니다
5. 아래 **Response body** 에 8.3 과 같은 JSON 이, **Code** 에 200 이 나옵니다

In [7]:
# 이 셀에서 하는 일 — Swagger 화면(/docs)을 노트북 안에 띄우고, 새 탭으로 열 링크도 만든다
# 왜 하나 — 코드 없이 버튼으로 API 에 요청을 보내 본다

if "google.colab" in sys.modules:                              # Colab 이면
    try:
        from google.colab import output                            # Colab 의 화면 도구
        링크 = output.eval_js(f"google.colab.kernel.proxyPort({포트})")   # 8765 번 서버를 브라우저에서 열 주소
        print("새 탭으로 열기 :", 링크 + "docs")
        output.serve_kernel_port_as_iframe(포트, path="/docs", height=600)   # 노트북 안에 바로 띄운다
    except Exception as 오류:                                     # 화면을 못 띄워도 아래 셀들은 계속 실행되게 한다
        print("Swagger 화면을 노트북 안에 띄우지 못했습니다 :", 오류)
else:                                                          # 내 컴퓨터면
    print(f"브라우저 주소창에 {주소}/docs 를 입력하세요")

브라우저 주소창에 http://localhost:8765/docs 를 입력하세요


**결과 읽는 법**

**결론: Swagger 화면에서 코드 없이 같은 요청을 보낼 수 있습니다. 화면에 나온 엔드포인트 두 개는 8.1 에서 읽은 것과 같습니다.**
- Colab 에서는 셀 아래에 **Swagger 화면**이 나타나고, 그 위에 **새 탭으로 열기** 링크가 찍힙니다
  - 화면이 작으면 링크를 눌러 새 탭에서 크게 보세요
  - 이 링크는 **노트북을 실행한 사람용**입니다. 다른 사람에게 보내도 열리지 않을 수 있으니 공유하지 마세요
- 화면 맨 위에 「추천 API」 이름과 설명, 그 아래 `GET /health` · `GET /recommend/{user_id}` 두 줄이 보입니다
- 화면이 나타나지 않으면 오류 문구가 대신 찍히고, 아래 셀은 정상으로 실행됩니다 (강사 화면으로 확인)

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 8.5 오류 응답 — 상태 코드와 입력 검증</mark>

**흐름에서의 위치:** 요청 → 서버 → **응답**. 요청이 잘못되었을 때 서버가 어떻게 응답하는지 봅니다.

**8.5 에서 하는 일** — 정상 요청 하나와 일부러 잘못된 요청 다섯 개를 보내고, 서버의 응답을 표로 비교합니다.

**왜 하나**
- API 는 늘 성공만 하지 않습니다. **잘못된 요청에 어떻게 응답할지**도 API 의 약속입니다.
- 답이 없을 때 빈 목록 `[]` 과 200 을 응답하면, 앱은 「추천할 게 없는 고객」인지 「번호가 틀렸는지」 구별할 수 없습니다.
- **상태 코드**로 「성공 / 요청이 잘못됨 / 서버 오류」를 나눠 주면, 앱이 상황에 맞는 화면(「고객 번호를 확인해 주세요」)을 보여 줄 수 있습니다.

**상태 코드 — 첫 숫자로 묶어서 기억합니다**

| 묶음 | 뜻 | 자주 보는 코드 |
|---|---|---|
| **2xx** | 성공 | **200** OK |
| **4xx** | **요청한 쪽**(클라이언트)의 문제 — 값 · 주소 · 메서드 · 인증 | **404** 찾는 것이 없음 · **405** 허용하지 않는 메서드 · **422** 요청 값이 규칙에 맞지 않음 |
| **5xx** | **서버 쪽**의 문제 | **500** 서버 안에서 오류 |

- 웹 서핑 중 「404 Not Found」 화면을 본 적이 있다면, 그것도 같은 404 입니다.
- 4xx 는 대체로 「요청을 고치면 해결」, 5xx 는 대체로 「서버를 고쳐야 해결」입니다.

**입력 검증 (validation)** — 서버가 요청 값을 **먼저 검사**하는 것입니다.
- 오늘 API 의 코드 `k: int = Query(10, ge=1, le=10)` 한 줄이 「`k` 는 정수이고 1 이상 10 이하」라는 규칙입니다. 검사하는 코드를 따로 쓰지 않아도 FastAPI 가 규칙에 맞지 않는 요청을 막고 422 로 응답합니다.
- 7주차의 데이터 점검 · 규격 검사와 같은 생각입니다 — 틀린 값이 안으로 들어오기 전에 막습니다.

**HTTP 메서드 — GET 과 POST**
- **GET** — 서버에서 정보를 **가져오기**(조회). 추천을 받는 일은 조회입니다
- **POST** — 서버에 데이터를 **보내서 처리하도록 요청하기**(저장 · 생성 등). 예 — 고객이 종목을 담았다는 기록을 서버에 보낼 때
- 이 API 는 GET 만 허용합니다. POST 로 요청하면 **405** 로 응답합니다

In [8]:
# 이 셀에서 하는 일 — 정상 요청 하나와 잘못된 요청 다섯 개를 보내고, 서버가 상태 코드로 어떻게 응답하는지 표로 본다
# 왜 하나 — 성공(200)과 오류(404 · 405 · 422)가 어떻게 다른지 직접 확인한다

시험_요청 = [                                                      # (무엇을 시험하나, 메서드, 주소)
    ("정상 요청",                          "GET",  "/recommend/U0001"),
    ("결과 파일에 없는 투자자 번호",         "GET",  "/recommend/U9999"),
    ("규칙을 어긴 k (10 초과)",             "GET",  "/recommend/U0001?k=20"),
    ("숫자가 아닌 k",                      "GET",  "/recommend/U0001?k=abc"),
    ("없는 주소",                         "GET",  "/nothing"),
    ("허용하지 않는 메서드 (POST)",          "POST", "/recommend/U0001"),
]

결과_표 = []
for 설명, 메서드, 경로 in 시험_요청:
    응답 = requests.request(메서드, 주소 + 경로)                    # 메서드를 바꿔 가며 요청한다
    본문 = 응답.json()
    if 응답.status_code == 200:
        요약 = f"추천 {len(본문['recommendations'])}개"
    elif isinstance(본문["detail"], list):                         # 422 는 detail 이 목록으로 온다
        요약 = 본문["detail"][0]["msg"]
    else:
        요약 = 본문["detail"]
    결과_표.append({"시험": 설명, "메서드": 메서드, "주소": 경로, "상태 코드": 응답.status_code, "응답 내용": 요약})

pd.DataFrame(결과_표)

,시험,메서드,주소,상태 코드,응답 내용
0,정상 요청,GET,/recommend/U0001,200,추천 10개
1,결과 파일에 없는 투자자 번호,GET,/recommend/U9999,404,U9999 는 추천이 없는 투자자입니다
2,규칙을 어긴 k (10 초과),GET,/recommend/U0001?k=20,422,Input should be less than or equal to 10
3,숫자가 아닌 k,GET,/recommend/U0001?k=abc,422,"Input should be a valid integer, unable to par..."
4,없는 주소,GET,/nothing,404,Not Found
5,허용하지 않는 메서드 (POST),POST,/recommend/U0001,405,Method Not Allowed


**결과 읽는 법**

**결론: 성공은 200, 잘못된 요청은 4xx 로 응답했고, 오류마다 이유를 함께 알려 줍니다. 빈 목록으로 얼버무리지 않습니다.**
- **404** (없는 번호 · 없는 주소) — 찾는 것이 없다
  - 번호 `U9999` — 결과 파일에 없어서 API 코드의 `HTTPException` 에 적어 둔 설명이 그대로 왔습니다
  - 없는 주소 `/nothing` — FastAPI 가 기본으로 `Not Found` 라고 응답합니다
- **422** (`k=20` · `k=abc`) — 요청 값이 규칙에 맞지 않다
  - `k=20` 은 「10 이하여야 한다」, `k=abc` 는 「정수여야 한다」입니다
  - 검사하는 줄을 코드에 따로 쓰지 않았는데 FastAPI 가 `Query(10, ge=1, le=10)` 을 보고 막았습니다
- **405** (POST) — 이 주소는 GET 만 허용합니다
- 모두 **4xx** 입니다. 요청을 고치면 해결되는 문제이고, 서버 오류(5xx)는 한 번도 나오지 않았습니다

**🗣 수업에서 함께 보기**

오늘 API 는 7주차 CSV 파일의 내용을 **그대로 응답할 뿐**입니다. 8.3 에서 확인했듯 내용도 파일과 똑같습니다.

그렇다면 고객 앱이 CSV 파일을 **직접 읽으면** 되지 않을까요? 왜 굳이 API 를 사이에 둘까요?

- 힌트 — 앱이 하나가 아니라 **여러 개**(모바일 앱 · 상담 화면 · 챗봇)라면? 그리고 CSV 파일에는 **300명 모두의** 추천이 들어 있다는 점을 생각해 보세요.
- 수업에서 함께 이야기합니다. 답은 노트북 맨 아래 **「답」** 에도 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 8.6 요청 더 읽기 — 경로 · 쿼리 매개변수와 응답 규격</mark>

> 📖 **혼자 읽기** — 복습할 때 직접 실행해 보는 부분입니다.

**8.6 에서 하는 일** — 두 가지를 확인합니다.
1. 주소에 값을 넣는 두 방법 — **경로 매개변수**와 **쿼리 매개변수**로 받는 개수를 바꿔 봅니다
2. **응답 규격**이 설명서(OpenAPI)에 자동으로 기록되는 것을 봅니다

**매개변수 (parameter) — 주소 안에 값을 넣는 두 방법**

| 종류 | 모양 | 오늘 예 | 언제 쓰나 |
|---|---|---|---|
| **경로 매개변수 (path parameter)** | 주소의 일부 | `/recommend/`**`U0001`** | **누구의 것인지**처럼 대상을 가리킬 때. 빼면 안 된다 |
| **쿼리 매개변수 (query parameter)** | 주소 뒤 `?이름=값` | `/recommend/U0001`**`?k=3`** | **몇 개 · 어떤 순서**처럼 선택 사항. 안 적어도 된다 (기본값 사용) |

- 여러 개는 `&` 로 잇습니다. 예: `?k=3&sort=score` (오늘 API 에는 `k` 만 있습니다)

**응답 규격 (response schema)**
- 응답의 **항목 이름과 형식**을 정해 둔 약속입니다. 7주차의 결과 파일 규격(output schema)과 같은 생각입니다.
- 오늘 API 코드의 `RecommendResponse` · `RecommendItem` 이 응답 규격입니다. 앱을 만드는 사람은 이것만 보고 응답을 읽는 코드를 쓸 수 있습니다.

In [9]:
# 이 셀에서 하는 일 — 쿼리 매개변수 k 를 바꿔 요청하고, 설명서(OpenAPI)에 기록된 응답 규격을 본다
# 왜 하나 — k 에 따라 응답이 달라지는 것과, 응답 규격이 코드에서 자동으로 만들어지는 것을 확인한다

for 주소_끝 in ["/recommend/U0001", "/recommend/U0001?k=1", "/recommend/U0001?k=3"]:
    응답 = requests.get(주소 + 주소_끝)
    print(f"{주소_끝:24s} → 상태 코드 {응답.status_code} · 추천 {len(응답.json()['recommendations'])}개")

설명서 = requests.get(주소 + "/openapi.json").json()                 # Swagger 가 읽는 설명서(OpenAPI)
규격 = 설명서["components"]["schemas"]
print("\n설명서에 기록된 응답 규격 :", [이름 for 이름 in 규격 if 이름.startswith("Recommend")])
print("RecommendResponse 의 항목  :", list(규격["RecommendResponse"]["properties"]))
print("RecommendItem 의 항목      :", list(규격["RecommendItem"]["properties"]))

/recommend/U0001         → 상태 코드 200 · 추천 10개
/recommend/U0001?k=1     → 상태 코드 200 · 추천 1개
/recommend/U0001?k=3     → 상태 코드 200 · 추천 3개

설명서에 기록된 응답 규격 : ['RecommendItem', 'RecommendResponse']
RecommendResponse 의 항목  : ['user_id', 'persona', 'recommendations']
RecommendItem 의 항목      : ['rank', 'item_id', 'item_name', 'risk_level', 'score']


**결과 읽는 법**

**결론: `k` 를 안 적으면 10개, `k=1` 이면 1개, `k=3` 이면 3개를 받습니다. 응답 규격은 코드에서 자동으로 설명서에 기록됩니다.**
- 경로 매개변수(`U0001`)는 반드시 적어야 하고, 쿼리 매개변수(`k`)는 안 적으면 기본값(10)을 씁니다
- 설명서에 응답 규격 두 개(`RecommendItem` · `RecommendResponse`)가 있고, 항목 이름이 코드와 같습니다
  - 코드를 바꾸면 설명서도 함께 바뀌므로, 설명서가 낡을 일이 없습니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 8.7 서빙 설계 — 일괄 추천과 실시간 추천, 지연 시간</mark>

> 📖 **혼자 읽기** — 복습할 때 직접 실행해 보는 부분입니다.

**8.7 에서 하는 일** — 추천을 응답하는 방법 세 가지의 **응답 시간**을 재 보고, 어느 방식을 고를지 생각합니다.

**왜 하나**
- 7주차에서 이름만 소개한 **일괄 추천 · 실시간 추천**을 직접 비교합니다. 추천 시스템을 서비스로 만들 때 가장 먼저 정하는 설계입니다.

**두 방식**

| | 일괄 추천 (batch) | 실시간 추천 (real-time) |
|---|---|---|
| 계산하는 때 | 요청이 오기 **전에** 모두 미리 | 요청이 **올 때** |
| 요청이 오면 | 만들어 둔 결과를 **찾아서 응답** | 그 자리에서 점수를 계산해 응답 |
| 응답 속도 | 빠르고 일정하다 | 계산 시간만큼 느리다 |
| 최신 기록 반영 | 다음 번 일괄 계산 때 | 요청 때 최신 기록을 쓰면 **바로** |
| 필요한 것 | 결과 파일 | 학습된 모델을 계속 켜 둔 서버 |
| 오늘의 API | **이 방식** (7주차 파일을 읽음) | — |

- 서비스에 따라 **섞어서** 쓰기도 합니다. 평소에는 일괄 추천을 응답하고, 방금 일어난 행동을 반영해야 할 때만 실시간으로 다시 계산합니다.

**지연 시간 (latency)** — 요청을 보낸 뒤 응답이 올 때까지 걸리는 시간입니다. 고객이 앱을 열 때마다 기다리는 시간이라서 짧을수록 좋습니다.

**세 가지를 재 봅니다**
1. **API 에 요청** (일괄 추천) — 만들어 둔 파일에서 찾아 응답. 네트워크로 요청을 주고받는 시간이 포함됩니다
2. **실시간 계산** — 요청이 올 때 `추천하기` 로 계산. 모델은 미리 학습해 둠
3. **학습까지 포함** — 요청이 올 때마다 학습부터 다시 (실제로는 하지 않는 방식)

In [10]:
# 이 셀에서 하는 일 — 추천을 응답하는 세 방식(API 요청 · 실시간 계산 · 학습까지 포함)의 평균 응답 시간을 잰다
# 왜 하나 — 일괄 추천과 실시간 추천의 지연 시간 차이를 숫자로 본다

import recsys                                                       # 7주차에서 쓴 도구 모음
import recommend_batch                                              # 7주차 스크립트의 함수를 불러 쓴다

items, users, interactions = recsys.load()
성향 = users.set_index("user_id")["persona"].to_dict()              # {투자자 번호: 투자 성향}
위험도 = items.set_index("item_id")["risk_level"].to_dict()         # {종목 번호: 위험도}
학습_결과 = recommend_batch.학습하기(interactions, items)            # 미리 학습해 둔 모델 (실시간 계산에 쓴다)

def api_요청():
    응답 = requests.get(주소 + "/recommend/U0001")                  # 1) 만들어 둔 파일에서 찾아 응답
    assert 응답.status_code == 200                                  # 실패한 요청의 시간을 성공처럼 재지 않는다

def 실시간_계산():
    recommend_batch.추천하기(학습_결과, "U0001", 성향, 위험도)        # 2) 요청이 올 때 점수를 계산 (모델은 미리 학습)

def 학습까지_포함():
    새_결과 = recommend_batch.학습하기(interactions, items)          # 3) 요청이 올 때마다 학습부터
    recommend_batch.추천하기(새_결과, "U0001", 성향, 위험도)

for 이름, 함수, 횟수 in [("1) API 에 요청 (일괄 추천)", api_요청, 100),
                       ("2) 실시간 계산 (모델은 미리 학습)", 실시간_계산, 100),
                       ("3) 요청마다 학습부터", 학습까지_포함, 10)]:
    시작 = time.perf_counter()
    for _ in range(횟수):
        함수()                                                      # 같은 일을 여러 번 하고 평균을 낸다
    평균_ms = (time.perf_counter() - 시작) / 횟수 * 1000
    print(f"{이름:28s} 평균 {평균_ms:8.2f} ms")

1) API 에 요청 (일괄 추천)          평균     2.47 ms
2) 실시간 계산 (모델은 미리 학습)        평균     0.02 ms


3) 요청마다 학습부터                 평균    55.05 ms


**결과 읽는 법**

**결론: 학습까지 포함한 3번이 가장 오래 걸립니다. 1번과 2번은 우리 데이터(종목 100개 · 투자자 300명)에서는 둘 다 빠릅니다.**
- 값은 컴퓨터마다 · 실행할 때마다 조금씩 다릅니다. 단위 `ms` 는 1,000분의 1초입니다
- **1번 API 요청** — 계산은 없고 파일에서 찾아 응답하지만, 요청과 응답을 **주고받는 시간**이 들어 있습니다
- **2번 실시간 계산** — 계산 자체는 아주 짧지만, 모델을 메모리에 계속 올려 둔 서버가 필요합니다
- **3번 학습부터** — 요청마다 학습하면 응답이 한참 느려집니다. 그래서 **학습은 가끔, 서빙은 자주**로 나눕니다 (7주차 7.6)
- 데이터가 커지면 계산 시간이 길어질 수 있습니다. 이 숫자는 우리 데이터의 예시이고, 실제 서비스의 선택은 최신성 · 비용 · 응답 시간 요구에 따라 달라집니다

**일괄 추천을 고르면 따라오는 문제들**
- **갱신** — API 는 켜질 때 파일을 한 번 읽습니다. 새 파일이 만들어져도 API 를 다시 켜야 반영됩니다
- **파일에 없는 투자자** — 오늘은 404 로 응답합니다. 실제 서비스에서는 인기 종목을 대신 응답하는 **대체 응답(fallback)** 을 쓰기도 합니다 (3주차의 콜드 스타트와 같은 문제)

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 8.8 금융에서 중요한 것 — 인증 · 권한 · 개인정보</mark>

> 📖 **혼자 읽기** — 복습할 때 직접 실행해 보는 부분입니다.

**8.8 에서 하는 일** — 오늘의 API 를 **인증 없이 인터넷에 열었다고 가정**하고, 투자자 번호를 차례로 바꿔 요청하면 어떤 일이 생기는지 직접 확인합니다.

**왜 하나**
- 금융 API 에서 가장 먼저 정해야 하는 것은 「누가 부를 수 있는가」입니다.
- 오늘 API 는 **투자자 번호만 알면 누구나** 그 사람의 추천과 투자 성향을 볼 수 있습니다. 투자 성향은 고객의 **개인정보**입니다.

**인증과 권한**

| | 확인하는 것 | 실패하면 |
|---|---|---|
| **인증 (authentication)** | 요청한 사람이 **누구인지** | **401** (인증이 안 됨) |
| **권한 (authorization)** | 그 사람이 **그 정보를 볼 수 있는지** (본인인가, 담당 직원인가) | **403** (권한이 없음) |

**실제 금융 API 가 하는 일**
- **API 키 (API key)** — 요청에 붙이는 확인용 문자열. 키가 없거나 틀리면 401 로 응답하도록 만들 수 있습니다
- **HTTPS** — 주고받는 내용을 **암호화**해서, 중간에서 내용을 엿보거나 바꾸기 어렵게 합니다. 요청한 사람을 확인하는 인증은 따로 필요합니다
- **최소한의 정보만 응답** — 응답에 꼭 필요한 항목만 넣습니다. 오늘 응답의 `persona` 가 정말 필요한지 따져 봅니다
- **요청 횟수 제한 (rate limit)** — 짧은 시간에 너무 많이 요청하면 막습니다
- **접속 기록 (log)** — 언제 어떤 요청이 왔고 결과가 어땠는지 남겨서, 이상한 요청을 찾아냅니다 (인증이 있으면 요청한 사람도 함께 남길 수 있습니다)

**아래 셀에서 확인하는 것** — 투자자 번호가 `U0001` 부터 `U0300` 까지 **차례대로** 붙어 있다는 점을 이용해 300번 요청하면, 인증이 없을 때 얼마나 쉽게 모두 알아낼 수 있는지 봅니다. (교육용 가상 데이터와 자신의 서버에만 하는 시험입니다. 실제 고객 데이터나 남의 서버에는 절대 하지 않습니다.)

In [11]:
# 이 셀에서 하는 일 — 투자자 번호를 U0001 부터 U0300 까지 차례로 요청해, 모든 투자자의 투자 성향을 모은다
# 왜 하나 — 인증이 없으면 번호만 바꿔 가며 요청해도 전체 고객의 개인정보를 얻을 수 있다는 것을 확인한다

성향_수 = {}                                                       # {투자 성향: 투자자 수}
시작 = time.perf_counter()
for 번호 in range(1, 301):
    사람 = f"U{번호:04d}"                                           # 1 → "U0001", 300 → "U0300"
    답 = requests.get(주소 + f"/recommend/{사람}?k=1").json()        # k=1 : 1개만 받아도 투자 성향이 함께 온다
    성향_수[답["persona"]] = 성향_수.get(답["persona"], 0) + 1
걸린_초 = time.perf_counter() - 시작

print(f"요청 300번 · {걸린_초:.1f}초")
print("알아낸 투자 성향 분포 :", 성향_수)

요청 300번 · 0.9초
알아낸 투자 성향 분포 : {'안정형': 100, '성장형': 100, '공격형': 100}


**결과 읽는 법**

**결론: 번호만 바꿔 가며 요청하면 몇 초 만에 300명 전원의 투자 성향을 알아낼 수 있습니다. 인증이 없는 금융 API 는 그대로 열 수 없습니다.**
- 요청 300번이 **몇 초** 안에 끝났고, 안정형 · 성장형 · 공격형 인원수가 모두 나왔습니다
- 투자자 번호가 **차례대로** 붙어 있어서 다음 번호를 쉽게 짐작할 수 있습니다
- 그래서 실제 서비스는 **인증**으로 요청한 사람을 확인하고, **권한**으로 본인(또는 담당 직원)의 정보만 응답하며, **요청 횟수를 제한**하고, **접속 기록**을 남깁니다
- 응답에 `persona` 가 들어 있으므로 이 요청은 추천 목록뿐 아니라 **투자 성향**이라는 개인정보도 함께 알려 줍니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 8.9 배포와 운영 — 이 컴퓨터 안의 API 를 인터넷으로</mark>

> 📖 **혼자 읽기** — 수업에서는 PPT 로 설명합니다. 오늘은 실습하지 않습니다.

**1. 배포 (deployment) 란**
- 지금 API 는 `localhost` — **이 Colab 런타임 안**에서만 부를 수 있습니다. Colab 런타임이 종료되면 API 도 꺼집니다.
- **배포**는 API 를 **늘 켜져 있는 서버 컴퓨터**에 올려, **인터넷 주소**로 부를 수 있게 하는 일입니다.

| | 지금 (`localhost`) | 배포한 뒤 |
|---|---|---|
| 누가 부를 수 있나 | 이 Colab 런타임 안의 프로그램만 | 접근이 허용된 프로그램 (인터넷 주소를 알아야 함) |
| 언제 켜져 있나 | Colab 런타임이 켜져 있는 동안만 | 보통 24시간 |
| 주소 | `http://localhost:8765` | 예: `https://api.회사이름.com` |

**2. 배포하는 곳 — 개념만**

| 방법 | 설명 |
|---|---|
| 회사 서버 | 회사가 가진 서버 컴퓨터에 올린다 |
| 클라우드 (cloud) | 빌려 쓰는 서버 컴퓨터에 올린다. 필요한 만큼 늘리고 줄일 수 있다 |
| 컨테이너 (container, 예 Docker) | API 코드와 필요한 도구를 **한 묶음**으로 포장해서, 어느 컴퓨터에서든 같은 모양으로 켜지게 한다 |

**3. 배포하면 함께 생기는 일**
- **결과 파일 갱신** — 매일 밤 7주차 스크립트가 새 파일을 만들면, API 가 새 파일을 다시 읽게 해야 합니다. 방법은 두 가지입니다
  - 스크립트가 끝난 뒤 API 를 **다시 켠다**(재시작)
  - API 가 정해진 시각마다 파일이 바뀌었는지 보고 **다시 읽는다**
- **상태 확인 (health check)** — `/health` 같은 주소로 서버가 켜져 있는지 계속 확인하고, 꺼졌으면 자동으로 다시 켜거나 담당자에게 알립니다
- **접속 기록 (log)** — 언제 어떤 요청이 왔고 어떤 상태 코드로 응답했는지 남깁니다(인증이 있으면 요청한 사람도 남길 수 있습니다). 404 · 422 가 갑자기 늘면 문제가 생겼다는 신호입니다
  - 오늘 서버도 `output/api_log.txt` 에 접속 기록을 남겼습니다. 아래 셀에서 봅니다
- **버전 관리** — 응답 모양을 바꾸면 기존 앱이 깨집니다. 그래서 `/v1/recommend` 처럼 **주소에 버전**을 넣어, 새 모양은 `/v2/` 로 따로 내보내는 방법을 씁니다

In [12]:
# 이 셀에서 하는 일 — 오늘 서버가 남긴 접속 기록(log)에서 오류 응답(404 · 405 · 422)만 골라 본다
# 왜 하나 — 서버가 요청과 결과를 기록으로 남기고, 오류 응답이 기록에서 눈에 띄는 것을 확인한다

with open("output/api_log.txt", encoding="utf-8") as 파일:
    기록 = 파일.read().splitlines()                                  # 줄 단위 목록

오류_기록 = []
for 줄 in 기록:
    if " 404 " in 줄 or " 405 " in 줄 or " 422 " in 줄:               # 오류 상태 코드가 들어 있는 줄
        오류_기록.append(줄)

print(f"접속 기록 전체 {len(기록)}줄 · 그중 오류 응답 {len(오류_기록)}줄")
for 줄 in 오류_기록:
    print(줄)

접속 기록 전체 418줄 · 그중 오류 응답 5줄
INFO:     127.0.0.1:57859 - "GET /recommend/U9999 HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:57861 - "GET /recommend/U0001?k=20 HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:57863 - "GET /recommend/U0001?k=abc HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:57865 - "GET /nothing HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:57867 - "POST /recommend/U0001 HTTP/1.1" 405 Method Not Allowed


**결과 읽는 법**

**결론: 서버가 요청마다 한 줄씩 기록을 남겼고, 8.5 에서 일부러 보낸 오류 요청이 상태 코드와 함께 그대로 남아 있습니다.**
- 각 줄에 요청 메서드 · 주소 · 상태 코드가 있습니다. 404 · 405 · 422 가 8.5 의 요청과 같습니다
- 요청한 사람은 기록에 나오지 않습니다. 오늘 API 에는 인증이 없기 때문입니다
- 실제 서비스에서는 이런 기록을 모아서, 오류 응답이 갑자기 늘면 알림을 보냅니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 8.10 오늘의 마무리</mark>

1. 7주차 결과 파일을 읽어 추천을 응답하는 **API** 를 FastAPI 로 만들었습니다 — 엔드포인트 `/health` · `/recommend/{user_id}`.
2. API 를 **서버**로 켜고, 이 노트북(**클라이언트**)에서 **GET 요청**을 보내 **200** 과 JSON 응답을 받았습니다.
3. 응답은 7주차 파일과 **순서까지 같았고**, **Swagger**(`/docs`)에서 코드 없이 버튼으로 같은 요청을 보낼 수 있었습니다.
4. 잘못된 요청에는 **404**(없음) · **405**(허용하지 않는 메서드) · **422**(규칙 위반) 로 응답했습니다. 4xx 는 대체로 요청 쪽(값 · 주소 · 메서드 · 인증)과 관련된 응답입니다.
5. **일괄 추천**은 만들어 둔 결과를 찾아 응답해서 빠르고, 갱신하려면 API 를 다시 켜야 합니다. **인증 · 권한** 확인이 없고 번호를 짐작할 수 있으면, 번호만 바꿔 다른 사람의 정보까지 볼 수 있습니다.

**오늘 기억할 한 줄** — **API 는 추천을 계산하지 않고, 만들어 둔 추천을 약속된 모양으로 응답합니다.**

다음 주에는 이 추천을 **사람이 보는 화면**(Streamlit 대시보드)으로 만들고, 투자자별로 비교 · 해석해 봅니다.

---

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📝 혼자 해 보기</mark>

수업이 끝난 뒤 혼자 풀어 보는 문제입니다. 문제마다 **어느 절의 결과를 보면 되는지** 제목에 적어 두었습니다. 답은 바로 아래 **「답」** 에 있습니다. 먼저 스스로 생각해 본 뒤에 읽어 보세요.

**📝 혼자 해 보기 1 · 8.5 결과**

U9999 처럼 없는 번호가 오면 API 는 **404 오류**로 응답합니다. 그냥 **빈 목록**(`"recommendations": []`)과 200 을 응답하면 안 될까요?

- 힌트 — 고객 앱 입장에서 「이 고객은 추천할 종목이 없다」와 「고객 번호가 잘못됐다」를 구별해야 할까요?

**📝 혼자 해 보기 2 · 8.1 코드**

API 는 **켜질 때 결과 파일을 한 번만** 읽습니다. 오늘 밤 7주차 스크립트가 새 결과 파일을 만들었다면, 이미 켜져 있는 API 는 새 추천을 응답할까요?

- 힌트 — 8.1 에서 읽은 코드의 「켜질 때 한 번 실행」 부분과, `recommend_api.py` 맨 위 설명의 마지막 줄을 읽어 보세요.

**📝 혼자 해 보기 3 · 8.7 일괄 추천과 실시간 추천**

상담 직원이 「방금 고객이 담은 종목」까지 반영한 추천을 **바로** 보고 싶어 합니다. 오늘의 일괄 추천 API 로 충분할까요? 어떻게 하면 좋을까요?

- 힌트 — 8.7 의 두 방식 표에서 「최신 기록 반영」 줄을 보세요.

**📝 혼자 해 보기 4 · 8.8 결과**

오늘 API 를 인증 없이 인터넷에 열었다고 해 봅시다. 8.8 에서 본 것처럼 번호를 차례로 바꿔 요청하면 어떤 문제가 생기고, 무엇으로 막을 수 있을까요?

- 힌트 — 응답에 `persona`(투자 성향)가 들어 있었습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">답</mark>

먼저 스스로 답을 생각해 본 뒤에 읽어 보세요. 답에 나오는 숫자는 모두 이 노트북을 실제로 실행해서 얻은 값입니다.

**함께 보기 답 — 여러 프로그램이 같은 추천을 받게 하고, 필요한 한 사람 것만 응답하고, 파일이 바뀌어도 앱을 고치지 않게 하려는 것입니다**

- **여러 프로그램이 같은 추천을 받습니다**
  - 모바일 앱 · 상담 화면 · 챗봇이 각자 파일을 읽으면, 누군가는 어제 파일을, 누군가는 오늘 파일을 읽을 수 있습니다
  - API 하나를 거치면 모두 **API 가 읽은 같은 결과 파일의 추천**을 받습니다
- **필요한 것만 응답합니다**
  - CSV 에는 **300명 모두**의 추천과 투자 성향이 들어 있습니다. 앱에 파일을 통째로 주면 다른 고객의 정보까지 넘어갑니다
  - API 는 요청한 **한 사람의 10줄만** 응답합니다
- **앱이 파일의 사정을 몰라도 됩니다**
  - 파일 이름 · 위치 · 열 순서가 바뀌거나 나중에 CSV 대신 데이터베이스를 쓰게 돼도, **주소와 응답의 모양만 그대로면** 앱은 고칠 필요가 없습니다
- **잘못된 요청을 한곳에서 막습니다** — 없는 번호는 404, 규칙 위반은 422 (8.5)
- **한 줄로 정리하면** — API 는 추천을 쓰는 프로그램들이 **같은 주소와 같은 응답 모양**으로 추천을 받게 하는 약속입니다.

**혼자 해 보기 1 답 — 구별해야 합니다. 둘은 고객에게 보여 줄 화면이 다르기 때문입니다**

- **「추천할 종목이 없다」** — 정상적인 상황입니다. 앱은 「오늘은 추천이 없습니다」나 인기 종목을 보여 주면 됩니다
- **「고객 번호가 잘못됐다」** — 어딘가 오류가 난 상황입니다. 앱은 「고객 정보를 확인해 주세요」를 보여 주고, 개발팀은 왜 틀린 번호가 왔는지 찾아야 합니다
- 빈 목록과 200 을 응답하면 두 상황이 **똑같아 보여서** 오류를 놓칩니다
- 404 로 나눠 주면 앱이 상황에 맞게 반응하고, 접속 기록에서 404 가 많아지면 문제를 빨리 알아챌 수 있습니다
- **한 줄로 정리하면** — 상태 코드는 「무엇이 잘못됐는지」를 알려 주는 약속입니다. 실패도 정확히 알려 줘야 합니다.

**혼자 해 보기 2 답 — 응답하지 않습니다. 켜질 때 읽은 옛 파일을 계속 씁니다. API 를 다시 켜야 새 추천이 나갑니다**

- `recommend_api.py` 의 `추천표 = pd.read_csv(...)` 는 함수 밖에 있어서 **API 가 켜질 때 한 번만** 실행됩니다
  - 그 뒤로는 메모리에 올려 둔 `추천표` 에서 찾을 뿐, 파일을 다시 열지 않습니다
- 그래서 파일이 새로 만들어져도 켜져 있는 API 는 **어제 추천**을 계속 응답합니다
- 실무에서는 보통 둘 중 하나를 합니다
  - 7주차 스크립트가 끝나면 **API 를 다시 켠다**(재시작)
  - API 가 정해진 시각마다 파일이 바뀌었는지 보고 **다시 읽는다**
- 이것도 파이프라인의 한 단계입니다 — 「결과 파일 저장」 다음에 「API 가 새 파일을 읽기」가 이어져야 합니다
- **한 줄로 정리하면** — 파일을 새로 만드는 것과, API 가 그 파일을 쓰게 하는 것은 별개의 단계입니다.

**혼자 해 보기 3 답 — 일괄 추천만으로는 부족합니다. 그 고객 한 명만 실시간으로 다시 계산하거나, 일괄 추천과 섞어 써야 합니다**

- 일괄 추천은 **다음 번 일괄 계산 때** 최신 기록이 반영됩니다. 방금 담은 종목은 아직 파일에 없습니다
- 방법은 두 가지입니다
  - **실시간 추천** — 요청이 올 때 그 고객의 최신 기록으로 점수를 다시 계산한다 (8.7 의 2번 방식). 모델을 켜 둔 서버가 필요합니다
  - **섞어 쓰기** — 평소에는 일괄 추천 파일을 응답하고, 방금 담은 종목만 추천에서 빼는 등 가벼운 보정만 실시간으로 한다
- 우리 데이터에서는 실시간 계산도 빠릅니다(8.7). 종목과 투자자가 훨씬 많아지면 계산 비용이 커질 수 있어서, 일괄 추천을 기본으로 하고 필요한 곳만 실시간으로 계산하는 경우가 많습니다
- **한 줄로 정리하면** — 최신성이 필요한 정도에 따라 일괄 추천과 실시간 추천을 고릅니다.

**혼자 해 보기 4 답 — 300명 전원의 투자 성향이 몇 초 만에 새어 나갑니다. 인증 · 권한 · 요청 횟수 제한 · 접속 기록으로 막습니다**

- 번호가 `U0001` 부터 `U0300` 까지 **차례대로** 붙어 있어서, 하나씩 바꿔 가며 요청하면 모두 알아낼 수 있습니다 (8.8 에서 확인)
- 응답에는 **투자 성향**(안정형 · 성장형 · 공격형)이 들어 있습니다. 고객이 설문으로 밝힌 금융 **개인정보**입니다
- 추천 목록으로 그 고객이 어떤 종목을 아직 담지 **않았는지**도 짐작할 수 있습니다 (이미 담은 종목은 추천에서 빠지므로)
- 그래서 실제 금융 API 는
  - **인증** — 요청한 사람이 누구인지 확인한다 (API 키 등). **HTTPS** 는 주고받는 내용을 암호화하는 별도의 보호입니다
  - **권한** — 그 고객 본인이거나 권한 있는 직원만 볼 수 있게 한다
  - **짐작하기 어려운 값**을 번호로 쓰거나 **요청 횟수를 제한**한다
  - **접속 기록**을 남겨 이상한 요청을 찾아낸다
- **한 줄로 정리하면** — API 를 인터넷에 여는 순간, 「누가 부를 수 있는가」가 가장 먼저 정해져야 합니다.